# 01 — Khám phá dữ liệu CSE-CIC-IDS2018

**Notebook gốc:** Bước 0, 1, EDA ở bước 2 và phần phân bố nhãn. Đọc/ghi và chuẩn hóa schema thực hiện qua `src/data_io.py`. **Notebook có thể chạy độc lập** sau khi chép Parquet vào `data/raw/`. Không huấn luyện mô hình ở đây.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Mở thư mục project trên VS Code, hoặc chạy notebook trong notebooks/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src import config as cfg
print("Project:", PROJECT_ROOT)
print("Dataset:", cfg.DATA_DIR)


Project: D:\network-anomaly-hybrid-vscode\network-anomaly-hybrid-vscode
Dataset: D:\network-anomaly-hybrid-vscode\network-anomaly-hybrid-vscode\data\raw


In [2]:
import pandas as pd
from src.spark_session import create_spark
from src.utils import day_of
from src.data_io import prepare_curated
from pyspark.sql import functions as F

files = cfg.TRAIN_FILES + [cfg.VAL_FILE] + cfg.TEST_FILES
missing = [str(p) for p in files if not p.exists()]
if missing:
    raise FileNotFoundError("Chưa có file dataset trong data/raw/:\n" + "\n".join(missing))
print("Train:", [day_of(p) for p in cfg.TRAIN_FILES]); print("Val:", day_of(cfg.VAL_FILE))
print("Test:", [day_of(p) for p in cfg.TEST_FILES])
spark = create_spark()
all_df = prepare_curated(spark, files)
print("Các cột:", all_df.columns)
print("Tổng bản ghi:", all_df.count())
(cfg.OUT_DIR / "metrics").mkdir(parents=True, exist_ok=True)


Train: ['2018-02-14', '2018-02-15']
Val: 2018-02-16
Test: ['2018-02-20', '2018-02-21', '2018-02-22', '2018-02-23', '2018-02-28', '2018-03-01', '2018-03-02']
Các cột: ['flow_duration', 'total_fwd_packets', 'total_backward_packets', 'fwd_packets_length_total', 'bwd_packets_length_total', 'fwd_packet_length_max', 'fwd_packet_length_min', 'fwd_packet_length_mean', 'fwd_packet_length_std', 'bwd_packet_length_max', 'bwd_packet_length_min', 'bwd_packet_length_mean', 'bwd_packet_length_std', 'flow_bytes_s', 'flow_packets_s', 'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min', 'fwd_iat_total', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max', 'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean', 'bwd_iat_std', 'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_length', 'bwd_header_length', 'fwd_packets_s', 'bwd_packets_s', 'packet_length_min', 'packet_length_max', 'packet_length_mean', 'packet_length_std', 'packet_length_variance', '

In [3]:
day_counts = all_df.groupBy("day").count().orderBy("day").toPandas()
labels = all_df.groupBy("day", "label_raw").count().orderBy("day", "label_raw").toPandas()
display(day_counts)
display(labels)
(cfg.OUT_DIR / "metrics").mkdir(parents=True, exist_ok=True)
labels.to_csv(cfg.OUT_DIR / "metrics" / "label_distribution.csv", index=False)


,day,count
0,2018-02-14,619346
1,2018-02-15,794812
2,2018-02-16,591873
3,2018-02-20,954846
4,2018-02-21,561396
5,2018-02-22,830224
6,2018-02-23,829405
7,2018-02-28,456873
8,2018-03-01,249170
9,2018-03-02,771587


,day,label_raw,count
0,2018-02-14,Benign,525245
1,2018-02-14,FTP-BruteForce,53
2,2018-02-14,SSH-Bruteforce,94048
3,2018-02-15,Benign,743498
4,2018-02-15,DoS attacks-GoldenEye,41406
5,2018-02-15,DoS attacks-Slowloris,9908
6,2018-02-16,Benign,446619
7,2018-02-16,DoS attacks-Hulk,145199
8,2018-02-16,DoS attacks-SlowHTTPTest,55
9,2018-02-20,Benign,379482


**Lưu ý:** Kết quả là thống kê dataset, không phải accuracy của mô hình. `prepare_curated()` có thể ghi dữ liệu lần đầu; các lần sau đặt `SKIP_INGEST=1` trước khi mở kernel để dùng lại.

In [4]:
spark.stop()